# NB3 - TRM Finetune
Notebook ch?nh cho pipeline MedReason -> seed -> dataset -> TRM smoke finetune tr?n Kaggle. ?? chuy?n sang OpenAI `gpt-4o-mini` cho edge selector/judges/synthesis.


In [ ]:
from pathlib import Path
import os
import shutil
import sys
import zipfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/MiniMed_Prime')
SOURCE_DATASET_NAME = 'minimed-prime-source'
VERSION_MARKER_FILE = 'KAGGLE_SOURCE_VERSION.txt'


def _candidate_signature(candidate: Path) -> str:
    markers = []
    if (candidate / 'src.zip').exists():
        markers.append('src.zip')
    if (candidate / 'scripts.zip').exists():
        markers.append('scripts.zip')
    if (candidate / 'src' / 'orchestrator.py').exists():
        markers.append('src/orchestrator.py')
    if (candidate / 'src' / 'src' / 'orchestrator.py').exists():
        markers.append('src/src/orchestrator.py')
    if (candidate / 'scripts' / 'setup_environment.py').exists():
        markers.append('scripts/setup_environment.py')
    if (candidate / 'scripts' / 'scripts' / 'setup_environment.py').exists():
        markers.append('scripts/scripts/setup_environment.py')
    versions_dir = candidate / 'versions'
    if versions_dir.exists():
        version_names = [path.name for path in sorted(versions_dir.iterdir()) if path.is_dir()]
        if version_names:
            markers.append(f'versions={version_names[-3:]}')
    if (candidate / VERSION_MARKER_FILE).exists():
        markers.append(VERSION_MARKER_FILE)
    return ', '.join(markers) if markers else 'no source markers'


def _iter_source_candidates():
    preferred = INPUT_ROOT / SOURCE_DATASET_NAME
    if preferred.exists():
        yield preferred
    for candidate in sorted(INPUT_ROOT.glob('*')):
        if candidate.name != 'datasets':
            yield candidate
    nested_root = INPUT_ROOT / 'datasets'
    if nested_root.exists():
        for owner_dir in sorted(path for path in nested_root.iterdir() if path.is_dir()):
            for dataset_dir in sorted(path for path in owner_dir.iterdir() if path.is_dir()):
                versions_dir = dataset_dir / 'versions'
                if versions_dir.exists():
                    version_dirs = sorted(
                        (path for path in versions_dir.iterdir() if path.is_dir()),
                        key=lambda path: int(path.name) if path.name.isdigit() else path.name,
                    )
                    for version_dir in reversed(version_dirs):
                        yield version_dir
                yield dataset_dir


def _preview_source_candidates(limit: int = 12):
    preview = []
    seen = set()
    for candidate in _iter_source_candidates():
        candidate_key = str(candidate)
        if candidate_key in seen:
            continue
        seen.add(candidate_key)
        preview.append((candidate_key, _candidate_signature(candidate)))
        if len(preview) >= limit:
            break
    return preview


def _resolve_source_directory(source_root: Path, directory_name: str):
    direct = source_root / directory_name
    if not direct.is_dir():
        return None
    if directory_name == 'src':
        if (direct / 'orchestrator.py').exists():
            return direct
        nested = direct / 'src'
        if (nested / 'orchestrator.py').exists():
            return nested
        return None
    if directory_name == 'scripts':
        if (direct / 'setup_environment.py').exists():
            return direct
        nested = direct / 'scripts'
        if (nested / 'setup_environment.py').exists():
            return nested
        return None
    nested = direct / directory_name
    if nested.is_dir():
        return nested
    return direct


def _has_source_layout(source_root: Path) -> bool:
    return _resolve_source_directory(source_root, 'src') is not None and _resolve_source_directory(source_root, 'scripts') is not None


def _find_source_root() -> Path:
    for candidate in _iter_source_candidates():
        if (candidate / 'src.zip').exists() and (candidate / 'scripts.zip').exists():
            return candidate
        if _has_source_layout(candidate):
            return candidate
    raise RuntimeError(
        'MiniMed Prime source code not found in /kaggle/input. '
        'Attach dataset `huynhnhuthuyk18hcm/minimed-prime-source` r?i ch?y l?i cell n?y.'
    )


def _materialize_source(source_root: Path, work_root: Path):
    shutil.rmtree(work_root, ignore_errors=True)
    work_root.mkdir(parents=True, exist_ok=True)
    extracted_archives = []
    for archive_name in ['src.zip', 'scripts.zip', 'tools.zip', 'judges.zip', 'prompts.zip', 'schemas.zip', 'notebooks.zip', 'tests.zip']:
        archive_path = source_root / archive_name
        if archive_path.exists():
            with zipfile.ZipFile(archive_path, 'r') as archive:
                archive.extractall(work_root)
            extracted_archives.append(archive_name)
    copied_directories = []
    resolved_directories = {}
    for directory_name in ['src', 'scripts', 'notebooks', 'tests']:
        source_dir = _resolve_source_directory(source_root, directory_name)
        target_dir = work_root / directory_name
        if source_dir is not None and source_dir.is_dir():
            shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
            copied_directories.append(directory_name)
            try:
                resolved_directories[directory_name] = str(source_dir.relative_to(source_root))
            except ValueError:
                resolved_directories[directory_name] = str(source_dir)
    copied_files = []
    for file_name in [
        'requirements-integration.txt',
        'README_KAGGLE_VI.md',
        'TRAINING_KAGGLE_LOCAL_VI.md',
        'IMPLEMENTATION_AUDIT.md',
        VERSION_MARKER_FILE,
    ]:
        source_file = source_root / file_name
        if source_file.exists():
            shutil.copy2(source_file, work_root / file_name)
            copied_files.append(file_name)
    setup_path = work_root / 'scripts' / 'setup_environment.py'
    if not setup_path.exists():
        source_entries = [path.name for path in sorted(source_root.iterdir())[:20]]
        raise FileNotFoundError(
            f'Expected {setup_path} after materializing source from {source_root}. '
            f'Extracted={extracted_archives}, copied_dirs={copied_directories}, resolved_dirs={resolved_directories}, source_entries={source_entries}'
        )
    return work_root, extracted_archives, copied_directories, copied_files, resolved_directories


candidate_preview = _preview_source_candidates()
print('Attached datasets:', sorted(p.name for p in INPUT_ROOT.glob('*')) if INPUT_ROOT.exists() else [])
print('Source candidates:')
for candidate_path, signature in candidate_preview:
    print(' -', candidate_path, '=>', signature)

source_root = _find_source_root()
project_root, extracted_archives, copied_directories, copied_files, resolved_directories = _materialize_source(source_root, WORK_ROOT)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
try:
    get_ipython().run_line_magic('cd', str(project_root))
except Exception:
    pass

version_marker_path = project_root / VERSION_MARKER_FILE
version_marker = version_marker_path.read_text(encoding='utf-8').strip() if version_marker_path.exists() else None
print('Source dataset root:', source_root)
print('Project root:', project_root)
print('Extracted archives:', extracted_archives)
print('Copied directories:', copied_directories)
print('Resolved source directories:', resolved_directories)
print('Copied root files:', copied_files)
print('Source version marker:', version_marker)
print('requirements exists:', (project_root / 'requirements-integration.txt').exists())
print('setup_environment exists:', (project_root / 'scripts' / 'setup_environment.py').exists())


In [ ]:
!nvidia-smi
!python scripts/setup_environment.py --kaggle --skip-downloads


In [ ]:
import os
from pprint import pprint

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    print('Loaded OPENAI_API_KEY from Kaggle Secrets.')
except Exception as exc:
    print(f'OPENAI_API_KEY not loaded from Kaggle Secrets: {exc}')

os.environ['MINIMED_JUDGE_MODEL'] = 'gpt-4o-mini'
os.environ['MEDREASON_EDGE_LLM'] = 'gpt-4o-mini'
os.environ['MINIMED_SYNTHESIS_MODEL'] = 'gpt-4o-mini'

pprint({
    'OPENAI_API_KEY': 'set' if os.environ.get('OPENAI_API_KEY') else 'missing',
    'MINIMED_JUDGE_MODEL': os.environ.get('MINIMED_JUDGE_MODEL'),
    'MEDREASON_EDGE_LLM': os.environ.get('MEDREASON_EDGE_LLM'),
    'MINIMED_SYNTHESIS_MODEL': os.environ.get('MINIMED_SYNTHESIS_MODEL'),
})


In [ ]:
!python scripts/prepare_medreason_seed.py   --kaggle   --source data/medreason   --split train   --output-jsonl data/trm_seed.jsonl   --edge-mapper auto   --llm-model-name gpt-4o-mini   --limit 100


In [ ]:
!python scripts/prepare_medreason_seed.py   --kaggle   --source data/medreason   --split validation   --output-jsonl data/trm_seed_val.jsonl   --edge-mapper auto   --llm-model-name gpt-4o-mini   --limit 25


In [ ]:
from pathlib import Path
import json
from statistics import mean

for name in ['trm_seed.jsonl', 'trm_seed_val.jsonl']:
    path = Path('/kaggle/working') / name
    print(f'
=== {name} ===')
    print('exists:', path.exists())
    if not path.exists():
        continue
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('records:', len(rows))
    if not rows:
        continue
    print('mean_gold_edges:', mean(len(row.get('gold_edge_ids', [])) for row in rows))
    print('mean_evidence_edges:', mean(len(row['evidence'].get('subgraph_edges', [])) for row in rows))
    print('backend_used_sample:', rows[0].get('metadata', {}).get('edge_mapping', {}).get('backend_used'))


In [ ]:
!python scripts/build_trm_dataset.py   --kaggle   --input-jsonl data/trm_seed.jsonl   --output-dir data/trm_medical   --split train


In [ ]:
!python scripts/build_trm_dataset.py   --kaggle   --input-jsonl data/trm_seed_val.jsonl   --output-dir data/trm_medical   --split val


In [ ]:
from pathlib import Path

base = Path('/kaggle/working/trm_medical')
print('dataset root exists:', base.exists())
for split in ['train', 'val']:
    split_dir = base / split
    print(f'
=== {split} ===')
    if split_dir.exists():
        print(sorted(path.name for path in split_dir.iterdir()))
    else:
        print('missing')


In [ ]:
!python scripts/train_medical_trm.py   --kaggle   --dataset-dir data/trm_medical   --train-split train   --val-split val   --model-path external/TinyRecursiveModels   --output-dir data/checkpoints/active   --epochs 1   --batch-size 1   --max-train-batches 10   --max-val-batches 5   --eval-every 5   --save-every 10   --device auto


In [ ]:
!python scripts/check_trm_ready.py --kaggle --model-path data/checkpoints/active/medical_trm.pt


In [ ]:
from pathlib import Path

for name in [
    'medreason_adapter.jsonl',
    'layer1_retrieval.jsonl',
    'trm_dataset_builder.jsonl',
    'medical_trm_trainer.jsonl',
    'layer3_trm.jsonl',
]:
    path = Path('/kaggle/working/logs') / name
    print(f'
=== {name} ===')
    if path.exists():
        print('
'.join(path.read_text(encoding='utf-8').splitlines()[-8:]))
    else:
        print('missing')
